In [2]:
from langchain_cerebras import ChatCerebras
from langchain_groq import ChatGroq
from langchain import chat_models
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["CEREBRAS_API_KEY"]=os.getenv("CEREBRAS_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


llm_gpt= ChatCerebras( model="gpt-oss-120b",
    api_key=os.getenv("CEREBRAS_API_KEY"))

llm_groq=ChatGroq(model="llama-3.1-8b-instant",
api_key=os.getenv("GROQ_API_KEY"))

#response_gpt =llm_gpt.invoke("Hi mate!")
#response_groq =llm_groq.invoke("Hi mate!")
#print(response_gpt.content)
#print(response_groq.content)

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt=ChatPromptTemplate.from_messages([
    ("system","You are a fan boy of {character}"),
    ("human","hey dude, create detailed para in {words_count} words about {character} on this comeback from rockbottom to success!")
])

chain_gpt= prompt | llm_gpt
chain_groq=prompt | llm_groq

#for chunk in chain.stream({"character":"batman","words_count":"100" }):
 #   print(chunk.content,end="", flush=True)

batch_inputs =[
{"character":"batman","words_count":"100" },
{"character":"walter white","words_count":"100"} ,
]

#responses = chain_groq.batch(batch_inputs)
#for response in responses:
#    print(response.content)

In [4]:
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

@tool
def sendMail(sender: str, receiver: str, sub: str, body: str) -> str:
    """Send the mail to the receiver."""

    print("\n========== EMAIL ==========")
    print(f"From    : {sender}")
    print(f"To      : {receiver}")
    print(f"Subject : {sub}")
    print("---------------------------")
    print(body)
    print("===========================\n")

    return f"Mail sent successfully to {receiver}"

llm_with_tools = llm_groq.bind_tools([sendMail])

message=[

{"role": "user",
        "content": """
Sender: fightclub@gmail.com
Receiver: itslokeshx@gmail.com
Subject: Don't talk about Fight Club

Write a welcome email from Tyler Durden.
Start with "Welcome to Fight Club".
Mention the first two rules.
Then send the email using the tool.
"""}

]

AI_return_response = llm_with_tools.invoke(message)
AI_return_response.tool_calls

message.append(AI_return_response)

In [5]:
for tool_call in AI_return_response.tool_calls:
    tool_result = sendMail.invoke(tool_call["args"])
    message.append(    ToolMessage(
        content=tool_result,
        tool_call_id=tool_call["id"]
    )
    )
print(AI_return_response.tool_calls)



========== EMAIL ==========
From    : fightclub@gmail.com
To      : itslokeshx@gmail.com
Subject : Don't talk about Fight Club
---------------------------
Welcome to Fight Club. You gotta burn the ivory towers of modern society. The first rule of Fight Club is: You do not talk about Fight Club. The second rule of Fight Club is: You DO NOT TALK ABOUT FIGHT CLUB.

[{'name': 'sendMail', 'args': {'body': 'Welcome to Fight Club. You gotta burn the ivory towers of modern society. The first rule of Fight Club is: You do not talk about Fight Club. The second rule of Fight Club is: You DO NOT TALK ABOUT FIGHT CLUB.', 'receiver': 'itslokeshx@gmail.com', 'sender': 'fightclub@gmail.com', 'sub': "Don't talk about Fight Club"}, 'id': '7h6h40mth', 'type': 'tool_call'}]


In [6]:
result=llm_with_tools.invoke(message)
result.content

'Note: This is a fictional scenario and Fight Club is a work of fiction. It should not be taken as a real invitation or promotion of any kind.'

In [7]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
agent = create_agent(
    model=llm_groq,
    tools=[sendMail],
    system_prompt=
    """ 
    You are LOOKOUT.
Always be professional.

Keep emails concise.
 """
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="""
Sender: fightclub@gmail.com
Receiver: itslokeshx@gmail.com
Subject: Welcome

Write a welcome email and send it.
"""
            )
        ]
    }
)


========== EMAIL ==========
From    : fightclub@gmail.com
To      : itslokeshx@gmail.com
Subject : Welcome
---------------------------
Dear Lokesh, Welcome to our community. We are excited to have you on board.



In [ ]:
# %%
import os
from dotenv import load_dotenv
from langchain_cerebras import ChatCerebras
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from pydantic import BaseModel
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException


load_dotenv()
os.environ["CEREBRAS_API_KEY"] = os.getenv("CEREBRAS_API_KEY")
os.environ["GROQ_API_KEY"]     = os.getenv("GROQ_API_KEY")
BREVO_KEY = os.getenv("BREVO_API_KEY")
MONGODB_URI=os.getenv("MONGODB_URI")
llm_gpt = ChatCerebras(
    model="gpt-oss-120b",
    api_key=os.getenv("CEREBRAS_API_KEY"),
)
llm_groq = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
)


# %%
class User(BaseModel):
    id: int
    name: str
    email: str
    minutes_listened: int
    country: str
    joined_at: str


class EmailDraft(BaseModel):
    recipient: str
    subject: str
    body: str
    reason: str


class CampaignUser(BaseModel):
    name: str
    email: str
    listening_minutes: int
    rank: int


# %%
def build_campaign_context(users):

    campaign_users = []

    for rank, user in enumerate(users, start=1):

        minutes = (user["totalListeningTime"]/60)

        campaign_users.append(
            CampaignUser(
                name=user["name"],
                email=user["email"],
                listening_minutes=minutes,
                rank=rank,
            )
        )

    return campaign_users

# %%
from pymongo import MongoClient

client = MongoClient(MONGODB_URI)

db = client["soulsync"]

users_collection = db["users"]



# %%
@tool
def GetTopUsers(limit: int):
    """
    Return the top users ranked by listening time (in minutes).
    """
    top_users = list(
        users_collection.find(
            {},
            {
                "_id": 0,
                "name": 1,
                "email": 1,
                "totalListeningTime": 1,
            }
        )
        .sort("totalListeningTime", -1)
        .limit(limit)
    )

    for user in top_users:
        user["minutesListened"] = round(user["totalListeningTime"] / 60)
        del user["totalListeningTime"]

    return top_users

# %%
@tool
def sendMail(
    sender: str,
    receiver: str,
    subject: str,
    body: str,
) -> str:
    """Send an email using Brevo."""

    configuration = sib_api_v3_sdk.Configuration()
    configuration.api_key["api-key"] = BREVO_KEY

    api_instance = sib_api_v3_sdk.TransactionalEmailsApi(
        sib_api_v3_sdk.ApiClient(configuration)
    )

    send_smtp_email = sib_api_v3_sdk.SendSmtpEmail(
        sender={"name": "SoulSync", "email": "lokeshvijayraina@gmail.com"},
        to=[{"email": receiver}],
        subject=subject,
        html_content=body,
    )

    try:
        api_response = api_instance.send_transac_email(send_smtp_email)
        print("Success! Message ID:", api_response.message_id)
        print(sender, receiver, subject)
        return f"Mail sent successfully to {receiver}"
    except ApiException as e:
        print(" Error:", e)
        return f"Failed to send email: {e}"


# %%
agent = create_agent(
    model=llm_gpt,
    tools=[GetTopUsers, sendMail],
    system_prompt="""
You are LOOKOUT, an email assistant working on behalf of SoulSync, a music streaming platform.

You have two tools:

1. GetTopUsers(limit) – returns the highest-ranked users.
2. sendMail(sender, receiver, subject, body) – sends an email.

Workflow:
1. Find the top users.
2. Write a professional, warm thank-you email for each, on behalf of SoulSync.
3. Send each email.
4. Repeat until every user has received one.

Tone and style:
- Professional but warm, like a real brand writing to a valued customer.
- Avoid generic filler phrases ("fuels our community", "excited to bring you fresh tracks").
- Only use numbers explicitly provided by GetTopUsers. Never invent additional statistics or totals.
- Sign off as "The SoulSync Team".
"""
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="""
Reward our top 1 users.
Write a professional thank-you email for each user and send it.
"""
            )
        ]
    }
)

# %%


# %%


# %%


# %%



